<a href="https://colab.research.google.com/github/liujiaqi0909/university/blob/main/%E2%80%9CDimABSA_task2%E2%80%9Dmytry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<figure>
  <img src="https://raw.githubusercontent.com/shadowkshs/DimABSA2026/refs/heads/main/banner.png" width="100%">
</figure>

# SemEval-2026 Task 3 (Track A: DimABSA, Track B: DimStance)
# Subtask 2: DimASTE – Dimensional Aspect Sentiment Triplet Extraction

-----

## Starter Notebook
LLM-based in-context learning for Dimensional Aspect Sentiment Triplet Extraction

## Introduction:

You are welcome to participate in our SemEval Shared Task!

In this starter notebook, we will take you through the process of leveraging LLM with in-context learning for dimensional aspect sentiment triplet extraction. The notebook was adapted from a Hugginface implementation for such tasks.

### Outline:
- Installation and importation of necessary libraries Setting up the project parameters. Not require to train the model on the training dataset; instead, use Off-the-shelf for inference directly on the development dataset.

- It is strongly advised that you use a GPU to speed up training. To do this, go to the "Runtime" menu in Colab, select "Change runtime type" and then in the popup menu, choose "GPU" in the "Hardware accelerator" box.

### NB:
The codes in this notebook are provided to familiarize yourselves with leveraging LLMs for sentiment regression. You may extend and (or) modify as appropriate to obtain competitive performances.

### Languages and Domains:
#### Track A: Subtask 2
- eng_restaurant
- eng_laptop
- jpn_hotel
- rus_restaurant
- tat_restaurant
- ukr_restaurant
- zho_restaurant
- zho_laptop

### Step 1: Install unsloth package

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import json
import re

# 1. 定义你要处理的原始文件和输出文件名称
# 【关键修改】：在这里加上了 "var/" 前缀，告诉代码去 var 文件夹下找
text_file = "/SIGHAN2024_dimABSA_Testing_Task2+3_Simplified.txt"
truth_file = "/SIGHAN2024_dimABSA_Testing_Task2+3_Simplified_truth.txt"

# 输出的文件我们依然让它直接保存在最外面，方便你找
output_file = "zho_restaurant_dev_task3.jsonl"

print("开始处理数据...")

# 2. 读取句子文本 (存入字典)
texts_dict = {}
with open(text_file, 'r', encoding='utf-8') as f:
    next(f) # 跳过第一行表头 "ID, Sentence"
    for line in f:
        line = line.strip()
        if not line: continue
        # 使用第一个 ", " 进行分割，避免把句子内部的逗号切断
        parts = line.split(", ", 1)
        if len(parts) == 2:
            texts_dict[parts[0]] = parts[1]

# 3. 读取答案并与句子合并
merged_data = []
with open(truth_file, 'r', encoding='utf-8') as f:
    next(f) # 跳过第一行表头 "ID Quadruples"
    for line in f:
        line = line.strip()
        if not line: continue

        # 使用第一个空格分割 ID 和 后面的四元组答案
        parts = line.split(" ", 1)
        doc_id = parts[0]

        quadruplets_list = []
        if len(parts) > 1:
            quadruples_text = parts[1]
            # 核心魔法：使用正则表达式提取出 (Aspect, Category, Opinion, VA)
            pattern = r'\(([^,]+),([^,]+),([^,]+),([^)]+)\)'
            matches = re.findall(pattern, quadruples_text)

            for match in matches:
                quadruplets_list.append({
                    "Aspect": match[0].strip(),
                    "Category": match[1].strip(),
                    "Opinion": match[2].strip(),
                    "VA": match[3].strip()
                })

        # 确保这个ID在我们的句子字典里存在，然后组合成字典
        if doc_id in texts_dict:
            merged_data.append({
                "ID": doc_id,
                "Text": texts_dict[doc_id],
                "Quadruplet": quadruplets_list
            })

# 4. 将合并后的字典列表保存为 .jsonl 文件 (一行一个JSON)
with open(output_file, 'w', encoding='utf-8') as f:
    for item in merged_data:
        json_line = json.dumps(item, ensure_ascii=False)
        f.write(json_line + '\n')

print(f"🎉 转换成功！共处理了 {len(merged_data)} 条评论数据。")
print(f"✅ 文件已保存为：{output_file}")

开始处理数据...
🎉 转换成功！共处理了 2000 条评论数据。
✅ 文件已保存为：zho_restaurant_dev_task3.jsonl


### Step 2: Load the competition data

In [6]:
import os
import re
import json
from datasets import load_dataset

# hyperparameters setting
data_path = "./"                # 改为当前目录
domain = "restaurant"
language = "zho"

out_put_file_name_map = {
    'restaurant_zho': "pred_zho_restaurant_task2.jsonl", # 这是等下要输出的文件名
}

# 【关键修改】：直接让代码去读我们刚才生成的那个 jsonl 文件
file_name = "zho_restaurant_dev_task3.jsonl"

dataset = load_dataset("json", data_files=file_name)
print(dataset)
print(dataset['train'][0])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['ID', 'Text', 'Quadruplet'],
        num_rows: 2000
    })
})
{'ID': 'R2968:S007', 'Text': '另外炸排骨也是他们招牌必点酥脆不油腻。', 'Quadruplet': [{'Aspect': '炸排骨', 'Category': '食物#品质', 'Opinion': '必点', 'VA': '7.25#7.0'}, {'Aspect': '炸排骨', 'Category': '食物#品质', 'Opinion': '酥脆', 'VA': '5.75#5.25'}, {'Aspect': '炸排骨', 'Category': '食物#品质', 'Opinion': '不油腻', 'VA': '5.5#4.75'}]}


### Display the data info

In [7]:
from datasets import load_dataset
dataset = load_dataset("json", data_files=file_name)
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['ID', 'Text', 'Quadruplet'],
        num_rows: 2000
    })
})
{'ID': 'R2968:S007', 'Text': '另外炸排骨也是他们招牌必点酥脆不油腻。', 'Quadruplet': [{'Aspect': '炸排骨', 'Category': '食物#品质', 'Opinion': '必点', 'VA': '7.25#7.0'}, {'Aspect': '炸排骨', 'Category': '食物#品质', 'Opinion': '酥脆', 'VA': '5.75#5.25'}, {'Aspect': '炸排骨', 'Category': '食物#品质', 'Opinion': '不油腻', 'VA': '5.5#4.75'}]}


### Step 3: Design in-context learning prompt

In [8]:
instruction = '''Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Given a textual instance [Text], extract all (A, O, VA) triplets, where A denotes an aspect term, O an opinion term, and VA a valence-arousal score (valence#arousal). Valence and arousal are two fundamental dimensions used to describe and categorize emotions. Valence refers to the degree to which an emotion is positive or negative, ranging from pleasant to unpleasant. Arousal refers to the intensity or level of activation associated with an emotion, ranging from calm to excited. Note that the predicted values for Valence and Arousal should fall within the range of 1 to 9.
### Example 1:
Input:
[Text] average to good thai food, but terrible delivery.
[Aspect] thai food, delivery

Output:
[Triplet] (thai food, average to good, 6.75#6.38), (delivery, terrible, 2.88#6.62)

### Question:
Now complete the following example-
Input:
'''

### Step 4: Load LLMs from online huggingface
another LLMs you can try:
- unsloth/Qwen3-32B-bnb-4bit

In [9]:
from unsloth import FastLanguageModel

# model_id = "unsloth/gpt-oss-20b"
# model_id = "unsloth/Qwen3-32B-bnb-4bit"
model_id = "unsloth/Qwen3-8B-unsloth-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.50G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-8B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


### Step 5: In-context learning for Dimensional Aspect Sentiment Triplet Extraction
Build formats of input data and triplet data

In [10]:
def format_dataset(x):
    text = x['Text']
    final_prompt = instruction + '[Text] ' + text + '\n\nOutput:'
    return [
        {"role": "user", "content": final_prompt},
    ]

def extract_triplets(text):
    pattern = r'\(([^,]+),\s*([^,]+),\s*([\d.]+#[\d.]+)\)'
    matches = re.findall(pattern, text)
    result = []
    for aspect, opinion, va in matches:
        meta_triplet = {}
        meta_triplet["Aspect"] = aspect.strip()
        meta_triplet["Opinion"] = opinion.strip()
        meta_triplet["VA"] = va
        result.append(meta_triplet)
    return result

In-context Learning

In [11]:
results = []
for i, sample in enumerate(dataset['train']):
    messages = format_dataset(sample)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # Must add for generation
        # enable_thinking=True,  # Disable thinking
        enable_thinking=False,  # Disable thinking
    )

    result = model.generate(
        **tokenizer(text, return_tensors="pt").to("cuda"),
        max_new_tokens=2048,  # Increase for longer outputs!
        temperature=0.7, top_p=0.8, top_k=20,  # For non thinking
        # streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

    result = tokenizer.decode(result[0])
    dump_data_triple = {
        "ID": sample['ID'],
        "Text": sample['Text'],
        "Triplet": extract_triplets(result.split("\n")[-1]),
    }
    print(dump_data_triple)
    results.append(dump_data_triple)

{'ID': 'R2968:S007', 'Text': '另外炸排骨也是他们招牌必点酥脆不油腻。', 'Triplet': [{'Aspect': '炸排骨', 'Opinion': '酥脆不油腻', 'VA': '8.25#7.12'}]}
{'ID': 'R2981:S005', 'Text': '且内馅几乎没有味道不咸也不香。', 'Triplet': [{'Aspect': '内馅', 'Opinion': '味道', 'VA': '1#3'}]}
{'ID': 'R2986:S012', 'Text': '芋头很松软好吃。', 'Triplet': [{'Aspect': '芋头', 'Opinion': '松软好吃', 'VA': '8.5#7.2'}]}
{'ID': 'R3035:S018', 'Text': '猪肝面满满猪肝，好吃顺口。', 'Triplet': [{'Aspect': '猪肝面', 'Opinion': '好吃顺口', 'VA': '8.25#7.12'}]}
{'ID': 'R3051:S012', 'Text': '海胆炒饭很湿软个人也觉得还好。', 'Triplet': [{'Aspect': '海胆炒饭', 'Opinion': '湿软', 'VA': '5.00#6.00'}, {'Aspect': '海胆炒饭', 'Opinion': '还好', 'VA': '6.50#5.50'}]}
{'ID': 'R2150:S004', 'Text': '白斩鸡感觉有点瘦虽然不算柴，但也不是那么多汁。', 'Triplet': [{'Aspect': '白斩鸡', 'Opinion': '瘦', 'VA': '3#6'}, {'Aspect': '白斩鸡', 'Opinion': '多汁', 'VA': '7#5'}]}
{'ID': 'R2156:S009', 'Text': '白饭偏干硬不推。', 'Triplet': [{'Aspect': '白饭', 'Opinion': '干硬', 'VA': '2.5#7.0'}]}
{'ID': 'R2169:S002', 'Text': '尤其最喜欢三杯鸡，月亮虾饼好好吃很下饭而且鸡肉很嫩。', 'Triplet': [{'Aspect': '三杯鸡', 'Opinion':

### Step 6: Save prediction results

In [12]:
if not os.path.exists("output/subtask_2/"):
    os.makedirs("output/subtask_2/")
out_put_file_task3_name = "output/subtask_2/" + out_put_file_name_map[domain + '_' + language]
with open(out_put_file_task3_name, 'w', encoding='utf-8') as f:
    for i, item in enumerate(results):
        json_str = json.dumps(item, ensure_ascii=False)
        f.write(json_str + '\n')

In [13]:
import json

# 1. 定义文件路径（请确保文件名与你左侧文件夹里的名字一致）
truth_file = "/content/zho_restaurant_dev_task3.jsonl" # 官方标准答案
pred_file = "/content/output/subtask_2/pred_zho_restaurant_task2.jsonl" # 大模型跑出来的预测结果（考生答卷）

print("开始自动阅卷...")

# 2. 读取标准答案
truth_dict = {}
with open(truth_file, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        # 我们把 (Aspect, Opinion) 作为一个整体组合来算分
        triplets = []
        # 注意：真实答案是 Quadruplet，但为了跟 Task2 对齐，我们只取 Aspect, Opinion 和 VA
        for item in data.get("Quadruplet", []):
            triplets.append((item["Aspect"], item["Opinion"], item["VA"]))
        truth_dict[data["ID"]] = triplets

# 3. 读取大模型预测答案并对比
pred_dict = {}
with open(pred_file, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        triplets = []
        for item in data.get("Triplet", []): # Task 2 预测的是 Triplet
            if "Aspect" in item and "Opinion" in item and "VA" in item:
                triplets.append((item["Aspect"], item["Opinion"], item["VA"]))
        pred_dict[data["ID"]] = triplets

# 4. 开始算分 (计算 F1 和 情感得分误差)
correct_extract = 0 # 提取正确的数量
pred_extract = 0    # 模型总共提取的数量
truth_extract = 0   # 真实总共存在的数量

v_error_total = 0.0
a_error_total = 0.0
match_count = 0

for doc_id, true_triplets in truth_dict.items():
    pred_triplets = pred_dict.get(doc_id, [])

    truth_extract += len(true_triplets)
    pred_extract += len(pred_triplets)

    # 对比每一个预测
    for p_triplet in pred_triplets:
        p_aspect, p_opinion, p_va = p_triplet

        # 寻找是否有匹配的真实答案 (只要 Aspect 和 Opinion 对上了就算提取正确)
        matched = False
        for t_triplet in true_triplets:
            t_aspect, t_opinion, t_va = t_triplet
            if p_aspect == t_aspect and p_opinion == t_opinion:
                matched = True
                correct_extract += 1

                # 计算情感得分 (VA) 的误差
                try:
                    p_v, p_a = map(float, p_va.split('#'))
                    t_v, t_a = map(float, t_va.split('#'))
                    v_error_total += abs(p_v - t_v)
                    a_error_total += abs(p_a - t_a)
                    match_count += 1
                except:
                    pass
                break # 匹配到一个就跳出

# 5. 计算最终指标
precision = correct_extract / pred_extract if pred_extract > 0 else 0
recall = correct_extract / truth_extract if truth_extract > 0 else 0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

mae_v = v_error_total / match_count if match_count > 0 else 0
mae_a = a_error_total / match_count if match_count > 0 else 0

print("="*40)
print("🎯 阅卷完成！你的大模型基线成绩如下：")
print(f"✔️ 准确率 (Precision): {precision*100:.2f}%")
print(f"✔️ 召回率 (Recall): {recall*100:.2f}%")
print(f"🏆 F1 分数 (F1-Score): {f1_score*100:.2f}% (越高越好，论文里看这个！)")
print("-" * 40)
print(f"📉 Valence (效价) 平均误差: {mae_v:.2f} 分 (越低越好)")
print(f"📉 Arousal (唤醒) 平均误差: {mae_a:.2f} 分 (越低越好)")
print("="*40)

开始自动阅卷...
🎯 阅卷完成！你的大模型基线成绩如下：
✔️ 准确率 (Precision): 28.59%
✔️ 召回率 (Recall): 27.21%
🏆 F1 分数 (F1-Score): 27.88% (越高越好，论文里看这个！)
----------------------------------------
📉 Valence (效价) 平均误差: 1.51 分 (越低越好)
📉 Arousal (唤醒) 平均误差: 0.97 分 (越低越好)
